# 1. Setup and Authentication
In this section, we install required libraries and authenticate with Hopsworks.

In [ ]:
!pip install -q hopsworks pandas numpy openmeteo-requests requests-cache retry-requests matplotlib seaborn scikit-learn xgboost shap

In [ ]:
import pandas as pd
import numpy as np
import hopsworks
import openmeteo_requests
import requests_cache
from retry_requests import retry
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Log in to Hopsworks
# Ensure you have your Hopsworks API key ready (you can add it to Kaggle Secrets)
import os
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# os.environ["HOPSWORKS_API_KEY"] = user_secrets.get_secret("HOPSWORKS_API_KEY")

project = hopsworks.login()
fs = project.get_feature_store()

# 2. Fetch Historical Data from OpenMeteo
We fetch historical weather and air quality data for Karachi (24.8607, 67.0011) from 2020 to present.

In [ ]:
def fetch_openmeteo_historical(lat, lon, start_date, end_date):
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    # 1. Weather Data
    weather_url = 'https://archive-api.open-meteo.com/v1/archive'
    weather_params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m']
    }
    weather_response = openmeteo.weather_api(weather_url, params=weather_params)[0]
    hourly_weather = weather_response.Hourly()

    weather_data = {
        'date': pd.date_range(
            start=pd.to_datetime(hourly_weather.Time(), unit='s', utc=True),
            end=pd.to_datetime(hourly_weather.TimeEnd(), unit='s', utc=True),
            freq=pd.Timedelta(seconds=hourly_weather.Interval()),
            inclusive='left'
        ),
        'temperature': hourly_weather.Variables(0).ValuesAsNumpy(),
        'humidity': hourly_weather.Variables(1).ValuesAsNumpy(),
        'precipitation': hourly_weather.Variables(2).ValuesAsNumpy(),
        'pressure': hourly_weather.Variables(3).ValuesAsNumpy(),
        'wind_speed': hourly_weather.Variables(4).ValuesAsNumpy(),
        'wind_direction': hourly_weather.Variables(5).ValuesAsNumpy()
    }
    df_weather = pd.DataFrame(data=weather_data)

    # 2. Air Quality Data
    aqi_url = 'https://air-quality-api.open-meteo.com/v1/air-quality'
    aqi_params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': ['pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'us_aqi']
    }
    aqi_response = openmeteo.weather_api(aqi_url, params=aqi_params)[0]
    hourly_aqi = aqi_response.Hourly()

    aqi_data = {
        'date': pd.date_range(
            start=pd.to_datetime(hourly_aqi.Time(), unit='s', utc=True),
            end=pd.to_datetime(hourly_aqi.TimeEnd(), unit='s', utc=True),
            freq=pd.Timedelta(seconds=hourly_aqi.Interval()),
            inclusive='left'
        ),
        'pm10': hourly_aqi.Variables(0).ValuesAsNumpy(),
        'pm2_5': hourly_aqi.Variables(1).ValuesAsNumpy(),
        'carbon_monoxide': hourly_aqi.Variables(2).ValuesAsNumpy(),
        'nitrogen_dioxide': hourly_aqi.Variables(3).ValuesAsNumpy(),
        'sulphur_dioxide': hourly_aqi.Variables(4).ValuesAsNumpy(),
        'ozone': hourly_aqi.Variables(5).ValuesAsNumpy(),
        'aqi': hourly_aqi.Variables(6).ValuesAsNumpy()
    }
    df_aqi = pd.DataFrame(data=aqi_data)

    # Merge
    df = pd.merge(df_weather, df_aqi, on='date', how='inner')
    df.dropna(inplace=True)
    return df

karachi_lat, karachi_lon = 24.8607, 67.0011
end_date = datetime.now().strftime('%Y-%m-%d')
df = fetch_openmeteo_historical(karachi_lat, karachi_lon, '2023-01-01', end_date)
df.head()

# 3. Feature Engineering & Target Creation
We create derived features (time-based features, moving averages) and our targets (AQI in 24h, 48h, 72h).

In [ ]:
def create_features_and_targets(df):
    df = df.copy()
    # Time-based features
    df['hour'] = df['date'].dt.hour
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    
    # Derived features (e.g., AQI change rate over 24h)
    df['aqi_24h_rolling_mean'] = df['aqi'].rolling(window=24).mean()
    df['aqi_change_rate'] = df['aqi'].pct_change(periods=24)
    
    # Create targets (predicting AQI at +24h, +48h, +72h)
    # We shift negative because we want the target column on row T to be the AQI at T+H
    df['target_aqi_24h'] = df['aqi'].shift(-24)
    df['target_aqi_48h'] = df['aqi'].shift(-48)
    df['target_aqi_72h'] = df['aqi'].shift(-72)
    
    df.dropna(inplace=True)
    # Format date as integer timestamp for Hopsworks primary key
    df['timestamp'] = df['date'].astype('int64') // 10**6
    return df

df_processed = create_features_and_targets(df)
print(df_processed.shape)
df_processed.tail()

# 4. Exploratory Data Analysis (EDA)
Visualize correlations to see which features impact AQI the most.

In [ ]:
plt.figure(figsize=(12, 10))
corr = df_processed.drop(['date', 'timestamp'], axis=1).corr()
sns.heatmap(corr, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

# 5. Save to Hopsworks Feature Store
We push this clean dataset to Hopsworks so our training pipeline can pull it.

In [ ]:
aqi_fg = fs.get_or_create_feature_group(
    name='aqi_features',
    version=1,
    primary_key=['timestamp'],
    event_time='timestamp',
    description='Karachi weather and AQI features'
)

aqi_fg.insert(df_processed)